In [ ]:
# ============================================================
# AI-Powered VaR Market Data Quality Control Cockpit
#
# BLOCK 1 - ENVIRONMENT SETUP
#
# This notebook demonstrates an end-to-end prototype for
# AI-assisted Market Data Quality Control for Value-at-Risk (VaR).
#
# This block:
# 1. Mounts Google Drive
# 2. Creates the project folders
# 3. Imports required libraries
# 4. Sets reproducibility controls
# 5. Performs environment quality checks
#
# Author: Hackathon Prototype
# ============================================================

#-------------------------------------------------------------
# Install required libraries
#-------------------------------------------------------------

!pip -q install plotly openai

#-------------------------------------------------------------
# Import libraries
#-------------------------------------------------------------

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px

from google.colab import drive

#-------------------------------------------------------------
# Reproducibility
#-------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

#-------------------------------------------------------------
# Mount Google Drive
#-------------------------------------------------------------

drive.mount('/content/drive')

#-------------------------------------------------------------
# Create Project Structure
#-------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Hackathon/var_control_cockpit"
)

folders = [
    "data/raw",
    "data/processed",
    "outputs",
    "audit",
    "reports",
    "models"
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(
        parents=True,
        exist_ok=True
    )

print("Project created at:")
print(PROJECT_ROOT)

#-------------------------------------------------------------
# Save configuration
#-------------------------------------------------------------

CONFIG = {

    "project_name":
        "AI Powered VaR Control Cockpit",

    "version":
        "0.1",

    "random_seed":
        RANDOM_SEED,

    "currency_pairs":[
        "USDSGD",
        "EURUSD",
        "GBPUSD",
        "USDJPY",
        "AUDUSD"
    ],

    "days_of_history":
        30,

    "sampling_frequency":
        "1H"

}

config_file = PROJECT_ROOT / "audit/config.json"

import json

with open(config_file,"w") as f:
    json.dump(CONFIG,f,indent=4)

#-------------------------------------------------------------
# Environment Quality Checks
#-------------------------------------------------------------

quality_checks = []

quality_checks.append({
    "Check":"Google Drive Mounted",
    "Passed":os.path.exists("/content/drive")
})

quality_checks.append({
    "Check":"Project Folder Exists",
    "Passed":PROJECT_ROOT.exists()
})

quality_checks.append({
    "Check":"Config File Saved",
    "Passed":config_file.exists()
})

quality_checks.append({
    "Check":"Pandas Loaded",
    "Passed":pd.__name__ == 'pandas'
})

quality_df = pd.DataFrame(quality_checks)

display(quality_df)

#-------------------------------------------------------------
# Stop execution if any quality check fails
#-------------------------------------------------------------

if not quality_df["Passed"].all():

    raise Exception(
        "Environment Quality Check Failed."
    )

#-------------------------------------------------------------
# Project Folder Overview
#-------------------------------------------------------------

folder_df = pd.DataFrame({

    "Folder":[
        str(PROJECT_ROOT / f)
        for f in folders
    ]

})

display(folder_df)

print("\n")

print("="*60)
print("BLOCK 1 COMPLETED SUCCESSFULLY")
print("="*60)

print("\nNext Block:")
print("Generate synthetic market observations,")
print("VaR scenarios,")
print("Portfolio exposures,")
print("Historical alerts.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project created at:
/content/drive/MyDrive/Hackathon/var_control_cockpit


,Check,Passed
0,Google Drive Mounted,True
1,Project Folder Exists,True
2,Config File Saved,True
3,Pandas Loaded,True


,Folder
0,/content/drive/MyDrive/Hackathon/var_control_c...
1,/content/drive/MyDrive/Hackathon/var_control_c...
2,/content/drive/MyDrive/Hackathon/var_control_c...
3,/content/drive/MyDrive/Hackathon/var_control_c...
4,/content/drive/MyDrive/Hackathon/var_control_c...
5,/content/drive/MyDrive/Hackathon/var_control_c...




BLOCK 1 COMPLETED SUCCESSFULLY

Next Block:
Generate synthetic market observations,
VaR scenarios,
Portfolio exposures,
Historical alerts.


In [ ]:
# ============================================================
# BLOCK 2
#
# Generate synthetic datasets for the VaR Control Cockpit
#
# Datasets:
# 1. Market observations
# 2. VaR scenarios
# 3. Portfolio exposures
# 4. Historical alerts
#
# Outputs are saved to Google Drive.
# ============================================================

from datetime import datetime

# -----------------------------------------------------------
# Configuration
# -----------------------------------------------------------

currency_pairs = CONFIG["currency_pairs"]

vendors = [
    "Reuters",
    "Bloomberg"
]

portfolios = [
    "FX Trading",
    "Treasury",
    "Corporate Banking",
    "Private Banking"
]

risk_levels = [
    "High",
    "Medium",
    "Low"
]

# -----------------------------------------------------------
# Generate hourly timestamps
# -----------------------------------------------------------

# FIX: Update sampling_frequency to 'h' as expected by pd.date_range
CONFIG["sampling_frequency"] = "h" # Temporary fix for this cell

timestamps = pd.date_range(
    end=pd.Timestamp.today(),
    periods=24*CONFIG["days_of_history"],
    freq=CONFIG["sampling_frequency"]
)

# -----------------------------------------------------------
# Base Prices
# -----------------------------------------------------------

base_price = {
    "USDSGD":1.35,
    "EURUSD":1.09,
    "GBPUSD":1.28,
    "USDJPY":155,
    "AUDUSD":0.67
}

market_rows = []

# -----------------------------------------------------------
# Generate Market Prices
# -----------------------------------------------------------

for pair in currency_pairs:

    current_price = base_price[pair]

    for ts in timestamps:

        # Small random walk

        current_price *= (
            1 + np.random.normal(0,0.001)
        )

        reuters = current_price

        bloomberg = current_price * (
            1 + np.random.normal(0,0.0003)
        )

        market_rows.append([
            ts,
            pair,
            reuters,
            bloomberg
        ])

market_df = pd.DataFrame(
    market_rows,
    columns=[
        "Timestamp",
        "RiskFactor",
        "ReutersPrice",
        "BloombergPrice"
    ]
)

# -----------------------------------------------------------
# Inject Stale Prices
# -----------------------------------------------------------

stale_index = np.random.choice(
    market_df.index,
    15,
    replace=False
)

market_df.loc[
    stale_index,
    "ReutersPrice"
] = market_df.loc[
    stale_index-1,
    "ReutersPrice"
].values

market_df["InjectedStale"] = False
market_df.loc[
    stale_index,
    "InjectedStale"
] = True

# -----------------------------------------------------------
# Inject Outliers
# -----------------------------------------------------------

outlier_index = np.random.choice(
    market_df.index,
    15,
    replace=False
)

market_df.loc[
    outlier_index,
    "ReutersPrice"
] *= np.random.uniform(
    1.05,
    1.10,
    len(outlier_index)
)

market_df["InjectedOutlier"] = False
market_df.loc[
    outlier_index,
    "InjectedOutlier"
] = True

# -----------------------------------------------------------
# Generate VaR Scenario Shocks
# -----------------------------------------------------------

scenario_df = pd.DataFrame({

    "ScenarioID":
        [f"S{i}" for i in range(1,11)],

    "Scenario":
        [
            "Interest Rate Shock",
            "FX Crisis",
            "Oil Spike",
            "Equity Crash",
            "China Slowdown",
            "Fed Rate Cut",
            "Inflation Spike",
            "Commodity Rally",
            "Geopolitical Event",
            "Pandemic"
        ],

    "ShockPercent":
        np.random.uniform(
            -8,
            8,
            10
        )

})

# -----------------------------------------------------------
# Portfolio Exposure
# -----------------------------------------------------------

exposure_rows = []

for portfolio in portfolios:

    for pair in currency_pairs:

        exposure_rows.append({

            "Portfolio":portfolio,

            "RiskFactor":pair,

            "ExposureUSD":
                np.random.randint(
                    5_000_000,
                    50_000_000
                ),

            "VaRContribution":
                round(
                    np.random.uniform(
                        1,
                        20
                    ),
                    2
                ),

            "Criticality":
                random.choice(
                    risk_levels
                )

        })

exposure_df = pd.DataFrame(exposure_rows)

# -----------------------------------------------------------
# Historical Alerts
# -----------------------------------------------------------

historical_df = pd.DataFrame({

    "AlertID":
        [f"A{i}" for i in range(1,51)],

    "RiskFactor":
        np.random.choice(
            currency_pairs,
            50
        ),

    "AlertType":
        np.random.choice(
            [
                "Stale",
                "Outlier",
                "Missing",
                "Source Divergence"
            ],
            50
        ),

    "ResolvedAs":
        np.random.choice(
            [
                "True Issue",
                "False Positive"
            ],
            50
        ),

    "RootCause":
        np.random.choice(
            [
                "Vendor Delay",
                "Market Movement",
                "Bad Mapping",
                "Holiday",
                "Missing Feed"
            ],
            50
        )

})

# -----------------------------------------------------------
# Save datasets
# -----------------------------------------------------------

market_df.to_csv(
    PROJECT_ROOT /
    "data/raw/market_data.csv",
    index=False
)

scenario_df.to_csv(
    PROJECT_ROOT /
    "data/raw/scenarios.csv",
    index=False
)

exposure_df.to_csv(
    PROJECT_ROOT /
    "data/raw/exposures.csv",
    index=False
)

historical_df.to_csv(
    PROJECT_ROOT /
    "data/raw/historical_alerts.csv",
    index=False
)

# -----------------------------------------------------------
# Quality Checks
# -----------------------------------------------------------

quality = pd.DataFrame({

    "Dataset":[
        "Market Data",
        "Scenario",
        "Exposure",
        "Historical Alerts"
    ],

    "Rows":[
        len(market_df),
        len(scenario_df),
        len(exposure_df),
        len(historical_df)
    ]

})

display(quality)

# -----------------------------------------------------------
# Visualisation
# -----------------------------------------------------------

sample_pair = "USDSGD"

plot_df = market_df[
    market_df["RiskFactor"]==sample_pair
]

fig = px.line(
    plot_df,
    x="Timestamp",
    y="ReutersPrice",
    title=f"{sample_pair} Market Price"
)

fig.show()

# -----------------------------------------------------------
# Show injected anomalies
# -----------------------------------------------------------

print()

print("Injected Stale Prices:",
      market_df["InjectedStale"].sum())

print("Injected Outliers:",
      market_df["InjectedOutlier"].sum())

print()

print("Datasets saved successfully.")

print("="*60)
print("BLOCK 2 COMPLETED")
print("="*60)

,Dataset,Rows
0,Market Data,3600
1,Scenario,10
2,Exposure,20
3,Historical Alerts,50



Injected Stale Prices: 15
Injected Outliers: 15

Datasets saved successfully.
BLOCK 2 COMPLETED


In [ ]:
# ============================================================
# BLOCK 3
#
# DATA VALIDATION & ENRICHMENT
#
# Purpose:
# 1. Load all generated datasets
# 2. Validate data quality
# 3. Create derived features
# 4. Simulate latency & market status
# 5. Save enriched dataset
# ============================================================

print("="*70)
print("BLOCK 3 - DATA VALIDATION & ENRICHMENT")
print("="*70)

# ----------------------------------------------------------
# Load datasets
# ----------------------------------------------------------

market_df = pd.read_csv(
    PROJECT_ROOT / "data/raw/market_data.csv",
    parse_dates=["Timestamp"]
)

scenario_df = pd.read_csv(
    PROJECT_ROOT / "data/raw/scenarios.csv"
)

exposure_df = pd.read_csv(
    PROJECT_ROOT / "data/raw/exposures.csv"
)

historical_df = pd.read_csv(
    PROJECT_ROOT / "data/raw/historical_alerts.csv"
)

print("Datasets loaded successfully\n")

# ----------------------------------------------------------
# Dataset Overview
# ----------------------------------------------------------

summary = pd.DataFrame({

    "Dataset":[
        "Market Data",
        "Scenario",
        "Exposure",
        "Historical Alerts"
    ],

    "Rows":[
        len(market_df),
        len(scenario_df),
        len(exposure_df),
        len(historical_df)
    ],

    "Columns":[
        market_df.shape[1],
        scenario_df.shape[1],
        exposure_df.shape[1],
        historical_df.shape[1]
    ]

})

display(summary)

# ----------------------------------------------------------
# Quality Checks
# ----------------------------------------------------------

quality = []

quality.append({

    "Check":"Duplicate Records",

    "Value":market_df.duplicated().sum(),

    "Expected":"0"

})

quality.append({

    "Check":"Missing Values",

    "Value":market_df.isna().sum().sum(),

    "Expected":"0"

})

quality.append({

    "Check":"Negative Prices",

    "Value":len(
        market_df[
            market_df["ReutersPrice"]<=0
        ]
    ),

    "Expected":"0"

})

quality.append({

    "Check":"Duplicate Timestamp/RiskFactor",

    "Value":market_df.duplicated(
        subset=["Timestamp","RiskFactor"]
    ).sum(),

    "Expected":"0"

})

quality_df = pd.DataFrame(quality)

display(quality_df)

# ----------------------------------------------------------
# Abort if serious issues found
# ----------------------------------------------------------

assert market_df["ReutersPrice"].min()>0,\
"Negative price detected"

# ----------------------------------------------------------
# Sort data
# ----------------------------------------------------------

market_df = market_df.sort_values(
    [
        "RiskFactor",
        "Timestamp"
    ]
)

# ----------------------------------------------------------
# Feature Engineering
# ----------------------------------------------------------

# Previous price

market_df["PreviousPrice"] = (
    market_df
    .groupby("RiskFactor")
    ["ReutersPrice"]
    .shift(1)
)

# Price Change

market_df["PriceChange"] = (
    market_df["ReutersPrice"]
    -
    market_df["PreviousPrice"]
)

# Percentage Change

market_df["PctChange"] = (
    market_df["PriceChange"]
    /
    market_df["PreviousPrice"]
)

# Rolling Mean

market_df["RollingMean"] = (

    market_df
    .groupby("RiskFactor")
    ["ReutersPrice"]
    .transform(
        lambda x:
        x.rolling(
            24,
            min_periods=5
        ).mean()
    )

)

# Rolling Std

market_df["RollingStd"] = (

    market_df
    .groupby("RiskFactor")
    ["ReutersPrice"]
    .transform(
        lambda x:
        x.rolling(
            24,
            min_periods=5
        ).std()
    )

)

# Z Score

market_df["ZScore"] = (

    market_df["ReutersPrice"]
    -
    market_df["RollingMean"]

) / market_df["RollingStd"]

# ----------------------------------------------------------
# Source Divergence
# ----------------------------------------------------------

market_df["SourceDifference"] = (

    abs(

        market_df["ReutersPrice"]

        -

        market_df["BloombergPrice"]

    )

)

market_df["SourceDifferencePct"] = (

    market_df["SourceDifference"]

    /

    market_df["BloombergPrice"]

)

# ----------------------------------------------------------
# Simulate Feed Latency
# ----------------------------------------------------------

market_df["FeedLatencySeconds"] = np.random.randint(

    0,
    600,
    len(market_df)

)

# ----------------------------------------------------------
# Simulate Market Open
# ----------------------------------------------------------

market_df["MarketOpen"] = np.where(

    market_df["Timestamp"].dt.dayofweek <5,

    True,

    False

)

# ----------------------------------------------------------
# Market Session
# ----------------------------------------------------------

hour = market_df["Timestamp"].dt.hour

market_df["MarketSession"] = np.select(

    [

        hour<8,

        hour<16,

        hour<22

    ],

    [

        "Asia",

        "Europe",

        "US"

    ],

    default="After Hours"

)

# ----------------------------------------------------------
# Join Exposure
# ----------------------------------------------------------

market_df = market_df.merge(

    exposure_df,

    on="RiskFactor",

    how="left"

)

# ----------------------------------------------------------
# Sample Output
# ----------------------------------------------------------

display(

    market_df.head(10)

)

# ----------------------------------------------------------
# Feature Summary
# ----------------------------------------------------------

feature_summary = pd.DataFrame({

    "Feature":[

        "Price Change",

        "Rolling Mean",

        "Rolling Std",

        "Z Score",

        "Source Difference",

        "Feed Latency",

        "Market Session",

        "Portfolio Exposure"

    ],

    "Purpose":[

        "Detect sudden movements",

        "Expected price",

        "Historical volatility",

        "Statistical anomaly",

        "Vendor disagreement",

        "Stale feed detection",

        "Trading hours context",

        "Business impact"

    ]

})

display(feature_summary)

# ----------------------------------------------------------
# Visualisation 1
# ----------------------------------------------------------

fig = px.histogram(

    market_df,

    x="ZScore",

    nbins=60,

    title="Distribution of Z Scores"

)

fig.show()

# ----------------------------------------------------------
# Visualisation 2
# ----------------------------------------------------------

fig = px.scatter(

    market_df.sample(800),

    x="SourceDifferencePct",

    y="FeedLatencySeconds",

    color="RiskFactor",

    title="Source Difference vs Feed Latency"

)

fig.show()

# ----------------------------------------------------------
# Save enriched dataset
# ----------------------------------------------------------

market_df.to_csv(

    PROJECT_ROOT /

    "data/processed/enriched_market_data.csv",

    index=False

)

print()

print("Enriched dataset saved.")

print()

print("Total Records :",len(market_df))

print("Average Z Score :",round(market_df["ZScore"].mean(),3))

print("Maximum Feed Latency :",market_df["FeedLatencySeconds"].max())

print()

print("="*70)
print("BLOCK 3 COMPLETED")
print("="*70)

BLOCK 3 - DATA VALIDATION & ENRICHMENT
Datasets loaded successfully



,Dataset,Rows,Columns
0,Market Data,3600,6
1,Scenario,10,3
2,Exposure,20,5
3,Historical Alerts,50,5


,Check,Value,Expected
0,Duplicate Records,0,0
1,Missing Values,0,0
2,Negative Prices,0,0
3,Duplicate Timestamp/RiskFactor,0,0


,Timestamp,RiskFactor,ReutersPrice,BloombergPrice,InjectedStale,InjectedOutlier,PreviousPrice,PriceChange,PctChange,RollingMean,...,ZScore,SourceDifference,SourceDifferencePct,FeedLatencySeconds,MarketOpen,MarketSession,Portfolio,ExposureUSD,VaRContribution,Criticality
0,2026-06-18 05:07:28.191944,AUDUSD,0.670020,0.670124,False,False,NaN,NaN,NaN,NaN,...,NaN,0.000104,0.000155,322,True,Asia,FX Trading,12903232,3.65,Medium
1,2026-06-18 05:07:28.191944,AUDUSD,0.670020,0.670124,False,False,NaN,NaN,NaN,NaN,...,NaN,0.000104,0.000155,322,True,Asia,Treasury,15339938,1.76,High
2,2026-06-18 05:07:28.191944,AUDUSD,0.670020,0.670124,False,False,NaN,NaN,NaN,NaN,...,NaN,0.000104,0.000155,322,True,Asia,Corporate Banking,40413058,10.52,Low
3,2026-06-18 05:07:28.191944,AUDUSD,0.670020,0.670124,False,False,NaN,NaN,NaN,NaN,...,NaN,0.000104,0.000155,322,True,Asia,Private Banking,14473103,3.77,High
4,2026-06-18 06:07:28.191944,AUDUSD,0.669362,0.669080,False,False,0.670020,-0.000658,-0.000981,NaN,...,NaN,0.000282,0.000422,473,True,Asia,FX Trading,12903232,3.65,Medium
5,2026-06-18 06:07:28.191944,AUDUSD,0.669362,0.669080,False,False,0.670020,-0.000658,-0.000981,NaN,...,NaN,0.000282,0.000422,473,True,Asia,Treasury,15339938,1.76,High
6,2026-06-18 06:07:28.191944,AUDUSD,0.669362,0.669080,False,False,0.670020,-0.000658,-0.000981,NaN,...,NaN,0.000282,0.000422,473,True,Asia,Corporate Banking,40413058,10.52,Low
7,2026-06-18 06:07:28.191944,AUDUSD,0.669362,0.669080,False,False,0.670020,-0.000658,-0.000981,NaN,...,NaN,0.000282,0.000422,473,True,Asia,Private Banking,14473103,3.77,High
8,2026-06-18 07:07:28.191944,AUDUSD,0.669434,0.669370,False,False,0.669362,0.000072,0.000107,NaN,...,NaN,0.000064,0.000096,231,True,Asia,FX Trading,12903232,3.65,Medium
9,2026-06-18 07:07:28.191944,AUDUSD,0.669434,0.669370,False,False,0.669362,0.000072,0.000107,NaN,...,NaN,0.000064,0.000096,231,True,Asia,Treasury,15339938,1.76,High


,Feature,Purpose
0,Price Change,Detect sudden movements
1,Rolling Mean,Expected price
2,Rolling Std,Historical volatility
3,Z Score,Statistical anomaly
4,Source Difference,Vendor disagreement
5,Feed Latency,Stale feed detection
6,Market Session,Trading hours context
7,Portfolio Exposure,Business impact



Enriched dataset saved.

Total Records : 14400
Average Z Score : -0.01
Maximum Feed Latency : 599

BLOCK 3 COMPLETED


In [ ]:
# ============================================================
# BLOCK 4
#
# STALE AND OUTLIER ALERT RULE ENGINE
#
# PURPOSE
# -------
# 1. Load the enriched market dataset.
# 2. Apply transparent market-data quality rules.
# 3. Identify stale, outlier, divergent, and delayed prices.
# 4. Assign alert severity.
# 5. Produce alert-level and portfolio-level outputs.
# 6. Validate the alert engine against injected anomalies.
# 7. Save the results to Google Drive.
#
# IMPORTANT DESIGN PRINCIPLE
# --------------------------
# This stage uses deterministic and explainable rules.
# Machine learning will be added in the next block as a
# complementary anomaly signal, not as a replacement for
# the bank's control rules.
# ============================================================


print("=" * 75)
print("BLOCK 4 - STALE AND OUTLIER ALERT RULE ENGINE")
print("=" * 75)


# ------------------------------------------------------------
# STEP 1: Load enriched market data
# ------------------------------------------------------------

ENRICHED_DATA_FILE = (
    PROJECT_ROOT
    / "data/processed/enriched_market_data.csv"
)

if not ENRICHED_DATA_FILE.exists():
    raise FileNotFoundError(
        "The enriched dataset was not found. "
        "Please run Block 3 before running Block 4."
    )

alerts_df = pd.read_csv(
    ENRICHED_DATA_FILE,
    parse_dates=["Timestamp"]
)

print(f"Loaded {len(alerts_df):,} enriched records.")
print()


# ------------------------------------------------------------
# STEP 2: Define configurable alert thresholds
# ------------------------------------------------------------
# These thresholds are illustrative for the hackathon.
# In production, they would be calibrated by:
#
# - asset class;
# - currency pair;
# - market session;
# - historical volatility;
# - control policy;
# - risk appetite.

ALERT_THRESHOLDS = {
    # A price is treated as unchanged when the relative
    # movement is smaller than this tolerance.
    "stale_price_tolerance_pct": 0.0000001,

    # Number of consecutive unchanged observations required
    # to classify a price as operationally stale.
    "stale_consecutive_periods": 2,

    # Large one-period movement threshold.
    "large_move_pct": 0.015,

    # Statistical outlier threshold.
    "absolute_z_score": 3.5,

    # Maximum permitted difference between Reuters and Bloomberg.
    "source_divergence_pct": 0.005,

    # Maximum accepted source latency.
    "maximum_latency_seconds": 300,

    # High-severity thresholds.
    "critical_z_score": 5.0,
    "critical_source_divergence_pct": 0.02,
    "critical_large_move_pct": 0.05
}

display(
    pd.DataFrame(
        {
            "Threshold": list(ALERT_THRESHOLDS.keys()),
            "Value": list(ALERT_THRESHOLDS.values())
        }
    )
)


# ------------------------------------------------------------
# STEP 3: Correct data ordering
# ------------------------------------------------------------
# Block 3 joined each market observation to multiple portfolios.
# Therefore, the same price observation can appear several times.
#
# To calculate stale runs correctly, we first calculate them on
# the unique market observation level and then merge the result
# back to the portfolio-level dataset.

alerts_df = alerts_df.sort_values(
    [
        "RiskFactor",
        "Timestamp",
        "Portfolio"
    ]
).reset_index(drop=True)


observation_columns = [
    "Timestamp",
    "RiskFactor",
    "ReutersPrice",
    "BloombergPrice",
    "InjectedStale",
    "InjectedOutlier",
    "PreviousPrice",
    "PctChange",
    "RollingMean",
    "RollingStd",
    "ZScore",
    "SourceDifference",
    "SourceDifferencePct",
    "FeedLatencySeconds",
    "MarketOpen",
    "MarketSession"
]


observation_df = (
    alerts_df[observation_columns]
    .drop_duplicates(
        subset=["Timestamp", "RiskFactor"]
    )
    .sort_values(
        ["RiskFactor", "Timestamp"]
    )
    .reset_index(drop=True)
)

print(
    "Unique market observations:",
    f"{len(observation_df):,}"
)

print(
    "Portfolio-enriched records:",
    f"{len(alerts_df):,}"
)


# ------------------------------------------------------------
# STEP 4: Calculate stale-price indicators
# ------------------------------------------------------------
# A single unchanged price may be legitimate.
# A sequence of unchanged prices is more suspicious.
#
# We calculate:
# - relative change from the previous observation;
# - whether the current observation is unchanged;
# - length of the current unchanged-price sequence.


observation_df["AbsolutePctChange"] = (
    observation_df["PctChange"].abs()
)


observation_df["PriceUnchanged"] = (
    observation_df["AbsolutePctChange"]
    <= ALERT_THRESHOLDS["stale_price_tolerance_pct"]
)


def calculate_consecutive_true_count(series):
    """
    Count consecutive True values.

    Example
    -------
    False, True, True, False, True
    becomes
    0, 1, 2, 0, 1
    """

    group_number = (~series).cumsum()

    return (
        series
        .groupby(group_number)
        .cumsum()
        .astype(int)
    )


observation_df["ConsecutiveUnchangedPeriods"] = (
    observation_df
    .groupby("RiskFactor")["PriceUnchanged"]
    .transform(calculate_consecutive_true_count)
)


observation_df["Rule_Stale"] = (
    observation_df["ConsecutiveUnchangedPeriods"]
    >= ALERT_THRESHOLDS["stale_consecutive_periods"]
)


# ------------------------------------------------------------
# STEP 5: Apply outlier and control rules
# ------------------------------------------------------------

# Large movement rule
observation_df["Rule_LargeMove"] = (
    observation_df["AbsolutePctChange"]
    >= ALERT_THRESHOLDS["large_move_pct"]
)


# Statistical outlier rule
observation_df["Rule_ZScoreOutlier"] = (
    observation_df["ZScore"].abs()
    >= ALERT_THRESHOLDS["absolute_z_score"]
)


# Cross-vendor source divergence
observation_df["Rule_SourceDivergence"] = (
    observation_df["SourceDifferencePct"]
    >= ALERT_THRESHOLDS["source_divergence_pct"]
)


# Source latency rule
observation_df["Rule_HighLatency"] = (
    observation_df["FeedLatencySeconds"]
    > ALERT_THRESHOLDS["maximum_latency_seconds"]
)


# Market closed indicator
# This is context rather than an alert by itself.
observation_df["Context_MarketClosed"] = (
    ~observation_df["MarketOpen"].astype(bool)
)


# ------------------------------------------------------------
# STEP 6: Count triggered rules
# ------------------------------------------------------------

RULE_COLUMNS = [
    "Rule_Stale",
    "Rule_LargeMove",
    "Rule_ZScoreOutlier",
    "Rule_SourceDivergence",
    "Rule_HighLatency"
]


observation_df["TriggeredRuleCount"] = (
    observation_df[RULE_COLUMNS]
    .sum(axis=1)
)


observation_df["HasRuleAlert"] = (
    observation_df["TriggeredRuleCount"] > 0
)


# ------------------------------------------------------------
# STEP 7: Create human-readable alert types
# ------------------------------------------------------------

RULE_LABELS = {
    "Rule_Stale": "Stale Price",
    "Rule_LargeMove": "Large Price Movement",
    "Rule_ZScoreOutlier": "Statistical Outlier",
    "Rule_SourceDivergence": "Source Divergence",
    "Rule_HighLatency": "High Feed Latency"
}


def build_alert_types(row):
    """
    Return a readable list of all rules triggered by one
    market observation.
    """

    triggered_alerts = [
        label
        for column, label in RULE_LABELS.items()
        if bool(row[column])
    ]

    if not triggered_alerts:
        return "No Alert"

    return " | ".join(triggered_alerts)


observation_df["AlertTypes"] = observation_df.apply(
    build_alert_types,
    axis=1
)


# ------------------------------------------------------------
# STEP 8: Calculate rule severity score
# ------------------------------------------------------------
# The rule score combines the number and seriousness of issues.
#
# This score is deliberately transparent:
#
# Stale price               = 25 points
# Large price movement      = 25 points
# Statistical outlier       = 25 points
# Source divergence         = 30 points
# High source latency       = 15 points
#
# Multiple simultaneous problems create a higher score.

observation_df["RuleSeverityScore"] = (
    observation_df["Rule_Stale"].astype(int) * 25
    + observation_df["Rule_LargeMove"].astype(int) * 25
    + observation_df["Rule_ZScoreOutlier"].astype(int) * 25
    + observation_df["Rule_SourceDivergence"].astype(int) * 30
    + observation_df["Rule_HighLatency"].astype(int) * 15
)


# Add extra weight for extreme conditions
observation_df["RuleSeverityScore"] += np.where(
    observation_df["ZScore"].abs()
    >= ALERT_THRESHOLDS["critical_z_score"],
    15,
    0
)

observation_df["RuleSeverityScore"] += np.where(
    observation_df["SourceDifferencePct"]
    >= ALERT_THRESHOLDS["critical_source_divergence_pct"],
    20,
    0
)

observation_df["RuleSeverityScore"] += np.where(
    observation_df["AbsolutePctChange"]
    >= ALERT_THRESHOLDS["critical_large_move_pct"],
    20,
    0
)


# Limit the score to 100 for easier interpretation
observation_df["RuleSeverityScore"] = (
    observation_df["RuleSeverityScore"]
    .clip(lower=0, upper=100)
)


# ------------------------------------------------------------
# STEP 9: Assign preliminary severity
# ------------------------------------------------------------
# This is the control-rule severity only.
#
# Scenario, VaR, exposure and P&L impact will be added later
# before calculating the final business priority.

severity_conditions = [
    observation_df["RuleSeverityScore"] >= 70,
    observation_df["RuleSeverityScore"] >= 45,
    observation_df["RuleSeverityScore"] >= 20,
    observation_df["RuleSeverityScore"] > 0
]

severity_labels = [
    "Critical",
    "High",
    "Medium",
    "Low"
]

observation_df["RuleSeverity"] = np.select(
    severity_conditions,
    severity_labels,
    default="No Alert"
)


# ------------------------------------------------------------
# STEP 10: Create unique observation-level alert IDs
# ------------------------------------------------------------

observation_df["ObservationID"] = (
    observation_df["RiskFactor"]
    + "_"
    + observation_df["Timestamp"].dt.strftime(
        "%Y%m%d_%H%M%S"
    )
)


observation_df["AlertID"] = np.where(
    observation_df["HasRuleAlert"],
    "RULE_"
    + observation_df["ObservationID"],
    None
)


# ------------------------------------------------------------
# STEP 11: Add alert evidence
# ------------------------------------------------------------
# Evidence fields allow reviewers and analysts to see exactly
# why the control triggered.

def build_rule_evidence(row):
    """
    Create a concise evidence statement for each alert.
    """

    evidence = []

    if row["Rule_Stale"]:
        evidence.append(
            "Price unchanged for "
            f"{int(row['ConsecutiveUnchangedPeriods'])} periods"
        )

    if row["Rule_LargeMove"]:
        evidence.append(
            "Absolute movement "
            f"{row['AbsolutePctChange']:.2%}"
        )

    if row["Rule_ZScoreOutlier"]:
        evidence.append(
            f"Z-score {row['ZScore']:.2f}"
        )

    if row["Rule_SourceDivergence"]:
        evidence.append(
            "Reuters-Bloomberg divergence "
            f"{row['SourceDifferencePct']:.2%}"
        )

    if row["Rule_HighLatency"]:
        evidence.append(
            "Feed latency "
            f"{int(row['FeedLatencySeconds'])} seconds"
        )

    if row["Context_MarketClosed"]:
        evidence.append(
            "Observation occurred while market was closed"
        )

    if not evidence:
        return "No control exception"

    return "; ".join(evidence)


observation_df["RuleEvidence"] = observation_df.apply(
    build_rule_evidence,
    axis=1
)


# ------------------------------------------------------------
# STEP 12: Merge rule results back to portfolio-level records
# ------------------------------------------------------------

rule_output_columns = [
    "Timestamp",
    "RiskFactor",
    "ObservationID",
    "AlertID",
    "AbsolutePctChange",
    "PriceUnchanged",
    "ConsecutiveUnchangedPeriods",
    "Rule_Stale",
    "Rule_LargeMove",
    "Rule_ZScoreOutlier",
    "Rule_SourceDivergence",
    "Rule_HighLatency",
    "Context_MarketClosed",
    "TriggeredRuleCount",
    "HasRuleAlert",
    "AlertTypes",
    "RuleSeverityScore",
    "RuleSeverity",
    "RuleEvidence"
]


alerts_df = alerts_df.merge(
    observation_df[rule_output_columns],
    on=["Timestamp", "RiskFactor"],
    how="left"
)


# ------------------------------------------------------------
# STEP 13: Create active alert datasets
# ------------------------------------------------------------

observation_alerts_df = observation_df[
    observation_df["HasRuleAlert"]
].copy()


portfolio_alerts_df = alerts_df[
    alerts_df["HasRuleAlert"]
].copy()


print()
print(
    "Observation-level alerts:",
    f"{len(observation_alerts_df):,}"
)

print(
    "Portfolio-level affected records:",
    f"{len(portfolio_alerts_df):,}"
)


# ------------------------------------------------------------
# STEP 14: Rule-engine quality gates
# ------------------------------------------------------------

quality_checks = []


def add_quality_check(
    check_name,
    actual_value,
    expected_condition,
    passed,
    explanation
):
    """
    Add one reviewable quality-gate result.
    """

    quality_checks.append(
        {
            "Quality Check": check_name,
            "Actual Value": actual_value,
            "Expected Condition": expected_condition,
            "Passed": bool(passed),
            "Explanation": explanation
        }
    )


# Gate 1: All rule columns must exist
missing_rule_columns = [
    column
    for column in RULE_COLUMNS
    if column not in observation_df.columns
]

add_quality_check(
    check_name="All alert rules created",
    actual_value=len(missing_rule_columns),
    expected_condition="0 missing rule columns",
    passed=len(missing_rule_columns) == 0,
    explanation=(
        "Confirms that all planned control checks were executed."
    )
)


# Gate 2: Every alert must have evidence
alerts_without_evidence = (
    observation_alerts_df["RuleEvidence"]
    .isna()
    .sum()
)

add_quality_check(
    check_name="Alert evidence coverage",
    actual_value=alerts_without_evidence,
    expected_condition="0 alerts without evidence",
    passed=alerts_without_evidence == 0,
    explanation=(
        "Every generated alert must explain why it triggered."
    )
)


# Gate 3: Every alert must have severity
alerts_without_severity = (
    observation_alerts_df["RuleSeverity"]
    .isin(["No Alert", ""])
    .sum()
)

add_quality_check(
    check_name="Alert severity coverage",
    actual_value=alerts_without_severity,
    expected_condition="0 alerts without severity",
    passed=alerts_without_severity == 0,
    explanation=(
        "Every alert must receive a preliminary control severity."
    )
)


# Gate 4: Injected outlier detection recall
injected_outliers = observation_df[
    observation_df["InjectedOutlier"].astype(bool)
]

detected_injected_outliers = injected_outliers[
    injected_outliers[
        [
            "Rule_LargeMove",
            "Rule_ZScoreOutlier",
            "Rule_SourceDivergence"
        ]
    ].any(axis=1)
]

outlier_recall = (
    len(detected_injected_outliers)
    / len(injected_outliers)
    if len(injected_outliers) > 0
    else 0
)

add_quality_check(
    check_name="Injected outlier detection recall",
    actual_value=f"{outlier_recall:.1%}",
    expected_condition="At least 75%",
    passed=outlier_recall >= 0.75,
    explanation=(
        "Measures whether the rule engine detects the synthetic "
        "outliers deliberately inserted in Block 2."
    )
)


# Gate 5: Any injected unchanged observation recognised
# Note: Block 2 injected isolated one-period repeats.
# Our formal stale rule requires two consecutive unchanged periods.
# Therefore, we separately verify recognition of unchanged prices.

injected_stale_rows = observation_df[
    observation_df["InjectedStale"].astype(bool)
]

recognised_unchanged = injected_stale_rows[
    injected_stale_rows["PriceUnchanged"]
]

unchanged_recognition_rate = (
    len(recognised_unchanged)
    / len(injected_stale_rows)
    if len(injected_stale_rows) > 0
    else 0
)

add_quality_check(
    check_name="Injected unchanged-price recognition",
    actual_value=f"{unchanged_recognition_rate:.1%}",
    expected_condition="At least 75%",
    passed=unchanged_recognition_rate >= 0.75,
    explanation=(
        "Confirms that deliberately repeated prices are recognised. "
        "The formal stale alert requires a longer sequence."
    )
)


# Gate 6: No duplicate observation alert IDs
duplicate_alert_ids = (
    observation_alerts_df["AlertID"]
    .duplicated()
    .sum()
)

add_quality_check(
    check_name="Unique observation alert IDs",
    actual_value=int(duplicate_alert_ids),
    expected_condition="0 duplicate alert IDs",
    passed=duplicate_alert_ids == 0,
    explanation=(
        "Each market observation alert must have a unique identifier."
    )
)


quality_df = pd.DataFrame(quality_checks)


display(
    quality_df.style.apply(
        lambda row: [
            (
                "background-color: #d4edda"
                if row["Passed"]
                else "background-color: #f8d7da"
            )
        ] * len(row),
        axis=1
    )
)


# Stop only if structural controls fail.
# Detection-recall quality checks remain visible for model tuning.

critical_check_names = [
    "All alert rules created",
    "Alert evidence coverage",
    "Alert severity coverage",
    "Unique observation alert IDs"
]

critical_failures = quality_df[
    quality_df["Quality Check"].isin(
        critical_check_names
    )
    & (~quality_df["Passed"])
]

if not critical_failures.empty:
    raise RuntimeError(
        "Critical alert-engine quality checks failed: "
        + ", ".join(
            critical_failures["Quality Check"].tolist()
        )
    )


# ------------------------------------------------------------
# STEP 15: Alert summary by rule
# ------------------------------------------------------------

rule_summary = pd.DataFrame(
    {
        "Alert Rule": [
            RULE_LABELS[column]
            for column in RULE_COLUMNS
        ],
        "Triggered Observations": [
            int(observation_df[column].sum())
            for column in RULE_COLUMNS
        ]
    }
).sort_values(
    "Triggered Observations",
    ascending=False
)

display(rule_summary)


# ------------------------------------------------------------
# STEP 16: Alert summary by severity
# ------------------------------------------------------------

severity_order = [
    "Critical",
    "High",
    "Medium",
    "Low"
]

severity_summary = (
    observation_alerts_df["RuleSeverity"]
    .value_counts()
    .reindex(
        severity_order,
        fill_value=0
    )
    .rename_axis("Severity")
    .reset_index(name="Alert Count")
)

display(severity_summary)


# ------------------------------------------------------------
# STEP 17: Alert summary by risk factor
# ------------------------------------------------------------

risk_factor_summary = (
    observation_alerts_df
    .groupby("RiskFactor")
    .agg(
        AlertCount=("AlertID", "count"),
        MaximumSeverityScore=(
            "RuleSeverityScore",
            "max"
        ),
        AverageZScore=(
            "ZScore",
            lambda values: values.abs().mean()
        ),
        MaximumSourceDivergence=(
            "SourceDifferencePct",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        "AlertCount",
        ascending=False
    )
)

display(risk_factor_summary)


# ------------------------------------------------------------
# STEP 18: Visualisation — alerts by rule
# ------------------------------------------------------------

fig = px.bar(
    rule_summary,
    x="Alert Rule",
    y="Triggered Observations",
    title="Market-Data Alerts by Control Rule",
    text="Triggered Observations"
)

fig.update_layout(
    xaxis_title="Control rule",
    yaxis_title="Number of observations"
)

fig.show()


# ------------------------------------------------------------
# STEP 19: Visualisation — alerts by severity
# ------------------------------------------------------------

fig = px.bar(
    severity_summary,
    x="Severity",
    y="Alert Count",
    title="Preliminary Alert Severity Distribution",
    category_orders={
        "Severity": severity_order
    },
    text="Alert Count"
)

fig.show()


# ------------------------------------------------------------
# STEP 20: Visualisation — alert timeline
# ------------------------------------------------------------

timeline_df = (
    observation_alerts_df
    .groupby(
        [
            pd.Grouper(
                key="Timestamp",
                freq="1D"
            ),
            "RuleSeverity"
        ]
    )
    .size()
    .reset_index(name="AlertCount")
)

fig = px.line(
    timeline_df,
    x="Timestamp",
    y="AlertCount",
    color="RuleSeverity",
    markers=True,
    title="Daily Alert Trend by Severity",
    category_orders={
        "RuleSeverity": severity_order
    }
)

fig.show()


# ------------------------------------------------------------
# STEP 21: Visualisation — outlier evidence
# ------------------------------------------------------------
# This scatter plot helps reviewers see how the rules behave.
#
# Points with high Z-score or high cross-vendor divergence
# should be visually distinct from ordinary observations.

plot_sample_size = min(
    2_000,
    len(observation_df)
)

outlier_plot_df = observation_df.sample(
    n=plot_sample_size,
    random_state=RANDOM_SEED
).copy()

fig = px.scatter(
    outlier_plot_df,
    x="ZScore",
    y="SourceDifferencePct",
    color="RuleSeverity",
    hover_data=[
        "Timestamp",
        "RiskFactor",
        "ReutersPrice",
        "BloombergPrice",
        "AlertTypes",
        "RuleEvidence"
    ],
    title=(
        "Statistical Outlier Score vs "
        "Cross-Vendor Price Divergence"
    ),
    category_orders={
        "RuleSeverity": [
            "Critical",
            "High",
            "Medium",
            "Low",
            "No Alert"
        ]
    }
)

fig.show()


# ------------------------------------------------------------
# STEP 22: Display highest-severity alert examples
# ------------------------------------------------------------

top_alert_columns = [
    "AlertID",
    "Timestamp",
    "RiskFactor",
    "ReutersPrice",
    "BloombergPrice",
    "PctChange",
    "ZScore",
    "SourceDifferencePct",
    "FeedLatencySeconds",
    "AlertTypes",
    "RuleSeverityScore",
    "RuleSeverity",
    "RuleEvidence"
]

top_alerts = (
    observation_alerts_df[
        top_alert_columns
    ]
    .sort_values(
        [
            "RuleSeverityScore",
            "Timestamp"
        ],
        ascending=[
            False,
            False
        ]
    )
    .head(15)
)

display(top_alerts)


# ------------------------------------------------------------
# STEP 23: Save outputs to Google Drive
# ------------------------------------------------------------

OBSERVATION_ALERT_FILE = (
    PROJECT_ROOT
    / "data/processed/rule_observation_alerts.csv"
)

PORTFOLIO_ALERT_FILE = (
    PROJECT_ROOT
    / "data/processed/rule_portfolio_alerts.csv"
)

FULL_RULE_OUTPUT_FILE = (
    PROJECT_ROOT
    / "data/processed/market_data_with_rules.csv"
)

QUALITY_REPORT_FILE = (
    PROJECT_ROOT
    / "reports/block4_rule_quality_checks.csv"
)

RULE_SUMMARY_FILE = (
    PROJECT_ROOT
    / "reports/block4_rule_summary.csv"
)


observation_alerts_df.to_csv(
    OBSERVATION_ALERT_FILE,
    index=False
)

portfolio_alerts_df.to_csv(
    PORTFOLIO_ALERT_FILE,
    index=False
)

alerts_df.to_csv(
    FULL_RULE_OUTPUT_FILE,
    index=False
)

quality_df.to_csv(
    QUALITY_REPORT_FILE,
    index=False
)

rule_summary.to_csv(
    RULE_SUMMARY_FILE,
    index=False
)


# ------------------------------------------------------------
# STEP 24: Final output summary
# ------------------------------------------------------------

total_observations = len(observation_df)
total_alerts = len(observation_alerts_df)

alert_rate = (
    total_alerts / total_observations
    if total_observations > 0
    else 0
)

print()
print("=" * 75)
print("BLOCK 4 COMPLETED SUCCESSFULLY")
print("=" * 75)

print(
    f"Unique market observations evaluated : "
    f"{total_observations:,}"
)

print(
    f"Observation-level alerts generated   : "
    f"{total_alerts:,}"
)

print(
    f"Alert rate                           : "
    f"{alert_rate:.2%}"
)

print(
    f"Affected portfolio records           : "
    f"{len(portfolio_alerts_df):,}"
)

print()
print("Files saved:")
print(f"1. {OBSERVATION_ALERT_FILE}")
print(f"2. {PORTFOLIO_ALERT_FILE}")
print(f"3. {FULL_RULE_OUTPUT_FILE}")
print(f"4. {QUALITY_REPORT_FILE}")
print(f"5. {RULE_SUMMARY_FILE}")

print()
print("Next block:")
print("Statistical anomaly detection using Isolation Forest.")
print("=" * 75)

BLOCK 4 - STALE AND OUTLIER ALERT RULE ENGINE
Loaded 14,400 enriched records.



,Threshold,Value
0,stale_price_tolerance_pct,1.000000e-07
1,stale_consecutive_periods,2.000000e+00
2,large_move_pct,1.500000e-02
3,absolute_z_score,3.500000e+00
4,source_divergence_pct,5.000000e-03
5,maximum_latency_seconds,3.000000e+02
6,critical_z_score,5.000000e+00
7,critical_source_divergence_pct,2.000000e-02
8,critical_large_move_pct,5.000000e-02


Unique market observations: 3,600
Portfolio-enriched records: 14,400

Observation-level alerts: 1,852
Portfolio-level affected records: 7,408


,Quality Check,Actual Value,Expected Condition,Passed,Explanation
0,All alert rules created,0,0 missing rule columns,True,Confirms that all planned control checks were executed.
1,Alert evidence coverage,0,0 alerts without evidence,True,Every generated alert must explain why it triggered.
2,Alert severity coverage,0,0 alerts without severity,True,Every alert must receive a preliminary control severity.
3,Injected outlier detection recall,100.0%,At least 75%,True,Measures whether the rule engine detects the synthetic outliers deliberately inserted in Block 2.
4,Injected unchanged-price recognition,100.0%,At least 75%,True,Confirms that deliberately repeated prices are recognised. The formal stale alert requires a longer sequence.
5,Unique observation alert IDs,0,0 duplicate alert IDs,True,Each market observation alert must have a unique identifier.


,Alert Rule,Triggered Observations
4,High Feed Latency,1840
1,Large Price Movement,30
3,Source Divergence,15
2,Statistical Outlier,14
0,Stale Price,0


,Severity,Alert Count
0,Critical,15
1,High,15
2,Medium,0
3,Low,1822


,RiskFactor,AlertCount,MaximumSeverityScore,AverageZScore,MaximumSourceDivergence
4,USDSGD,387,100,1.081286,0.090283
3,USDJPY,382,100,1.053804,0.097678
0,AUDUSD,368,15,1.214735,0.002437
2,GBPUSD,363,100,1.011330,0.099959
1,EURUSD,352,100,1.039296,0.092995


,AlertID,Timestamp,RiskFactor,ReutersPrice,BloombergPrice,PctChange,ZScore,SourceDifferencePct,FeedLatencySeconds,AlertTypes,RuleSeverityScore,RuleSeverity,RuleEvidence
1309,RULE_EURUSD_20260712_180728,2026-07-12 18:07:28.191944,EURUSD,1.153942,1.077644,0.071758,4.677325,0.070800,449,Large Price Movement | Statistical Outlier | S...,100,Critical,Absolute movement 7.18%; Z-score 4.68; Reuters...
1892,RULE_GBPUSD_20260707_010728,2026-07-07 01:07:28.191944,GBPUSD,1.384557,1.258735,0.100288,4.658949,0.099959,207,Large Price Movement | Statistical Outlier | S...,100,Critical,Absolute movement 10.03%; Z-score 4.66; Reuter...
1861,RULE_GBPUSD_20260705_180728,2026-07-05 18:07:28.191944,GBPUSD,1.367170,1.272289,0.072240,4.596015,0.074575,557,Large Price Movement | Statistical Outlier | S...,100,Critical,Absolute movement 7.22%; Z-score 4.60; Reuters...
3218,RULE_USDSGD_20260702_070728,2026-07-02 07:07:28.191944,USDSGD,1.462719,1.341595,0.089645,4.672544,0.090283,146,Large Price Movement | Statistical Outlier | S...,100,Critical,Absolute movement 8.96%; Z-score 4.67; Reuters...
1760,RULE_GBPUSD_20260701_130728,2026-07-01 13:07:28.191944,GBPUSD,1.326393,1.253235,0.058115,2.529575,0.058375,364,Large Price Movement | Source Divergence | Hig...,100,Critical,Absolute movement 5.81%; Reuters-Bloomberg div...
1750,RULE_GBPUSD_20260701_030728,2026-07-01 03:07:28.191944,GBPUSD,1.362310,1.248036,0.092646,4.689158,0.091563,375,Large Price Movement | Statistical Outlier | S...,100,Critical,Absolute movement 9.26%; Z-score 4.69; Reuters...
2373,RULE_USDJPY_20260627_020728,2026-06-27 02:07:28.191944,USDJPY,168.724291,153.923504,0.095952,4.657151,0.096157,68,Large Price Movement | Statistical Outlier | S...,100,Critical,Absolute movement 9.60%; Z-score 4.66; Reuters...
1645,RULE_GBPUSD_20260626_180728,2026-06-26 18:07:28.191944,GBPUSD,1.342476,1.229113,0.092964,4.677322,0.092231,419,Large Price Movement | Statistical Outlier | S...,100,Critical,Absolute movement 9.30%; Z-score 4.68; Reuters...
911,RULE_EURUSD_20260626_040728,2026-06-26 04:07:28.191944,EURUSD,1.193754,1.092186,0.092536,4.664601,0.092995,584,Large Price Movement | Statistical Outlier | S...,100,Critical,Absolute movement 9.25%; Z-score 4.66; Reuters...
1608,RULE_GBPUSD_20260625_050728,2026-06-25 05:07:28.191944,GBPUSD,1.325048,1.236342,0.069806,4.676743,0.071749,513,Large Price Movement | Statistical Outlier | S...,100,Critical,Absolute movement 6.98%; Z-score 4.68; Reuters...



BLOCK 4 COMPLETED SUCCESSFULLY
Unique market observations evaluated : 3,600
Observation-level alerts generated   : 1,852
Alert rate                           : 51.44%
Affected portfolio records           : 7,408

Files saved:
1. /content/drive/MyDrive/Hackathon/var_control_cockpit/data/processed/rule_observation_alerts.csv
2. /content/drive/MyDrive/Hackathon/var_control_cockpit/data/processed/rule_portfolio_alerts.csv
3. /content/drive/MyDrive/Hackathon/var_control_cockpit/data/processed/market_data_with_rules.csv
4. /content/drive/MyDrive/Hackathon/var_control_cockpit/reports/block4_rule_quality_checks.csv
5. /content/drive/MyDrive/Hackathon/var_control_cockpit/reports/block4_rule_summary.csv

Next block:
Statistical anomaly detection using Isolation Forest.


In [ ]:
# ============================================================
# BLOCK 5
#
# AI Statistical Anomaly Detection
#
# This block uses Isolation Forest to identify unusual market
# observations.
#
# Unlike rule-based detection, Isolation Forest does not rely
# on predefined thresholds.
#
# Instead, it learns what "normal" market behaviour looks like
# and identifies observations that appear statistically unusual.
# ============================================================

# Fix for ImportError: cannot import name '_center' from 'numpy._core.umath'
# This typically indicates a version mismatch between numpy, scipy, and scikit-learn.
# Reinstalling them will ensure compatible versions are used.
!pip install --upgrade scikit-learn scipy numpy

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

print("="*70)
print("BLOCK 5 - AI ANOMALY DETECTION")
print("="*70)

# ------------------------------------------------------------
# Load rule output
# ------------------------------------------------------------

market_df = pd.read_csv(

    PROJECT_ROOT /
    "data/processed/market_data_with_rules.csv",

    parse_dates=["Timestamp"]

)

print("Loaded records:",len(market_df))

# ------------------------------------------------------------
# Features used by AI
# ------------------------------------------------------------

features = [

    "PctChange",

    "ZScore",

    "SourceDifferencePct",

    "FeedLatencySeconds"

]

print("\nFeatures used by Isolation Forest")

display(pd.DataFrame({

    "Feature":features,

    "Purpose":[

        "Price movement",

        "Historical deviation",

        "Vendor disagreement",

        "Feed latency"

    ]

}))

# ------------------------------------------------------------
# Handle missing values
# ------------------------------------------------------------

X = market_df[features].copy()

X = X.fillna(0)

# ------------------------------------------------------------
# Standardise features
# ------------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# ------------------------------------------------------------
# Train Isolation Forest
# ------------------------------------------------------------

model = IsolationForest(

    contamination=0.03,

    random_state=42,

    n_estimators=200

)

model.fit(X_scaled)

# ------------------------------------------------------------
# Predict anomalies
# ------------------------------------------------------------

market_df["AIPrediction"] = model.predict(X_scaled)

# Convert prediction

market_df["AIAnomaly"] = np.where(

    market_df["AIPrediction"]==-1,

    True,

    False

)

# Raw anomaly score

market_df["AIScore"] = -model.score_samples(X_scaled)

# ------------------------------------------------------------
# Scale score to 0-100
# ------------------------------------------------------------

min_score = market_df["AIScore"].min()

max_score = market_df["AIScore"].max()

market_df["AISeverityScore"] = (

    (market_df["AIScore"]-min_score)

    /(max_score-min_score)

)*100

market_df["AISeverityScore"] = market_df["AISeverityScore"].round(1)

# ------------------------------------------------------------
# AI Severity
# ------------------------------------------------------------

market_df["AISeverity"] = pd.cut(

    market_df["AISeverityScore"],

    bins=[0,40,60,80,100],

    labels=[

        "Low",

        "Medium",

        "High",

        "Critical"

    ],

    include_lowest=True

)

# ------------------------------------------------------------
# Combined Alert
# ------------------------------------------------------------

market_df["CombinedAlert"] = np.where(

    (market_df["HasRuleAlert"])

    |

    (market_df["AIAnomaly"]),

    True,

    False

)

# ------------------------------------------------------------
# Explain AI Result
# ------------------------------------------------------------

def explain_ai(row):

    reasons=[]

    if abs(row["ZScore"])>3:

        reasons.append("Extreme Z Score")

    if row["SourceDifferencePct"]>0.005:

        reasons.append("Vendor disagreement")

    if row["FeedLatencySeconds"]>300:

        reasons.append("High latency")

    if abs(row["PctChange"])>0.015:

        reasons.append("Large movement")

    if len(reasons)==0:

        reasons.append("Multivariate anomaly")

    return ", ".join(reasons)

market_df["AIExplanation"] = market_df.apply(

    explain_ai,

    axis=1

)

# ------------------------------------------------------------
# Quality Checks
# ------------------------------------------------------------

quality=[]

quality.append({

    "Check":"Model trained",

    "Passed":True

})

quality.append({

    "Check":"AI scores generated",

    "Passed":

    market_df["AISeverityScore"].notna().all()

})

quality.append({

    "Check":"Combined Alert created",

    "Passed":

    market_df["CombinedAlert"].notna().all()

})

quality_df=pd.DataFrame(quality)

display(quality_df)

# ------------------------------------------------------------
# AI Summary
# ------------------------------------------------------------

summary=pd.DataFrame({

    "Metric":[

        "Total Records",

        "Rule Alerts",

        "AI Anomalies",

        "Combined Alerts"

    ],

    "Value":[

        len(market_df),

        market_df["HasRuleAlert"].sum(),

        market_df["AIAnomaly"].sum(),

        market_df["CombinedAlert"].sum()

    ]

})

display(summary)

# ------------------------------------------------------------
# Top AI anomalies
# ------------------------------------------------------------

top = market_df[

    market_df["AIAnomaly"]

].sort_values(

    "AISeverityScore",

    ascending=False

).head(20)

display(

top[

[
"Timestamp",
"RiskFactor",
"ReutersPrice",
"ZScore",
"SourceDifferencePct",
"FeedLatencySeconds",
"AISeverityScore",
"AISeverity",
"AIExplanation"
]

]

)

# ------------------------------------------------------------
# Visualisation
# ------------------------------------------------------------

fig=px.scatter(

    market_df.sample(1500),

    x="ZScore",

    y="AISeverityScore",

    color="AISeverity",

    hover_data=[

        "RiskFactor",

        "AIExplanation"

    ],

    title="AI Detected Statistical Anomalies"

)

fig.show()

# ------------------------------------------------------------
# Rule vs AI
# ------------------------------------------------------------

comparison=pd.DataFrame({

    "Detection":[

        "Rule",

        "AI",

        "Both"

    ],

    "Count":[

        market_df["HasRuleAlert"].sum(),

        market_df["AIAnomaly"].sum(),

        len(

            market_df[

                (market_df["HasRuleAlert"])

                &

                (market_df["AIAnomaly"])

            ]

        )

    ]

})

fig=px.bar(

comparison,

x="Detection",

y="Count",

text="Count",

title="Rule Engine vs AI Detection"

)

fig.show()

# ------------------------------------------------------------
# Save output
# ------------------------------------------------------------

market_df.to_csv(

PROJECT_ROOT/

"data/processed/market_data_with_ai.csv",

index=False

)

print()

print("="*70)

print("BLOCK 5 COMPLETED")

print("="*70)

print()

print("AI detected",

market_df["AIAnomaly"].sum(),

"statistical anomalies.")

print()

print("Next Block:")

print("Market-Level Comparison")

BLOCK 5 - AI ANOMALY DETECTION
Loaded records: 14400

Features used by Isolation Forest


,Feature,Purpose
0,PctChange,Price movement
1,ZScore,Historical deviation
2,SourceDifferencePct,Vendor disagreement
3,FeedLatencySeconds,Feed latency


,Check,Passed
0,Model trained,True
1,AI scores generated,True
2,Combined Alert created,True


,Metric,Value
0,Total Records,14400
1,Rule Alerts,7408
2,AI Anomalies,432
3,Combined Alerts,7556


,Timestamp,RiskFactor,ReutersPrice,ZScore,SourceDifferencePct,FeedLatencySeconds,AISeverityScore,AISeverity,AIExplanation
6580,2026-06-26 18:07:28.191944,GBPUSD,1.342476,4.677322,0.092231,419,100.0,Critical,"Extreme Z Score, Vendor disagreement, High lat..."
6581,2026-06-26 18:07:28.191944,GBPUSD,1.342476,4.677322,0.092231,419,100.0,Critical,"Extreme Z Score, Vendor disagreement, High lat..."
6582,2026-06-26 18:07:28.191944,GBPUSD,1.342476,4.677322,0.092231,419,100.0,Critical,"Extreme Z Score, Vendor disagreement, High lat..."
6583,2026-06-26 18:07:28.191944,GBPUSD,1.342476,4.677322,0.092231,419,100.0,Critical,"Extreme Z Score, Vendor disagreement, High lat..."
7003,2026-07-01 03:07:28.191944,GBPUSD,1.362310,4.689158,0.091563,375,99.5,Critical,"Extreme Z Score, Vendor disagreement, High lat..."
7002,2026-07-01 03:07:28.191944,GBPUSD,1.362310,4.689158,0.091563,375,99.5,Critical,"Extreme Z Score, Vendor disagreement, High lat..."
7001,2026-07-01 03:07:28.191944,GBPUSD,1.362310,4.689158,0.091563,375,99.5,Critical,"Extreme Z Score, Vendor disagreement, High lat..."
7000,2026-07-01 03:07:28.191944,GBPUSD,1.362310,4.689158,0.091563,375,99.5,Critical,"Extreme Z Score, Vendor disagreement, High lat..."
7571,2026-07-07 01:07:28.191944,GBPUSD,1.384557,4.658949,0.099959,207,99.3,Critical,"Extreme Z Score, Vendor disagreement, Large mo..."
7570,2026-07-07 01:07:28.191944,GBPUSD,1.384557,4.658949,0.099959,207,99.3,Critical,"Extreme Z Score, Vendor disagreement, Large mo..."



BLOCK 5 COMPLETED

AI detected 432 statistical anomalies.

Next Block:
Market-Level Comparison


In [ ]:
# ============================================================
# BLOCK 6
#
# MARKET CONTEXT & EXPLAINABILITY
#
# PURPOSE
# -------
# 1. Understand whether an alert is isolated or market-wide
# 2. Compare against peer instruments
# 3. Calculate market movement statistics
# 4. Find historical similarity
# 5. Produce analyst recommendations
#
# Output:
# market_data_with_context.csv
# ============================================================

print("="*70)
print("BLOCK 6 - MARKET CONTEXT")
print("="*70)

# ----------------------------------------------------------
# Load AI output
# ----------------------------------------------------------

market_df = pd.read_csv(
    PROJECT_ROOT /
    "data/processed/market_data_with_ai.csv",
    parse_dates=["Timestamp"]
)

historical_df = pd.read_csv(
    PROJECT_ROOT /
    "data/raw/historical_alerts.csv"
)

print("Loaded records:", len(market_df))

# ----------------------------------------------------------
# Create unique observation dataset
# ----------------------------------------------------------

obs_cols = [
    "Timestamp",
    "RiskFactor",
    "ReutersPrice",
    "BloombergPrice",
    "PctChange",
    "ZScore",
    "SourceDifferencePct",
    "FeedLatencySeconds",
    "HasRuleAlert",
    "AIAnomaly",
    "RuleSeverityScore",
    "AISeverityScore",
    "AlertTypes"
]

obs_df = (
    market_df[obs_cols]
    .drop_duplicates(
        subset=["Timestamp","RiskFactor"]
    )
)

print("Unique market observations:",len(obs_df))

# ----------------------------------------------------------
# Overall Market Movement
# ----------------------------------------------------------

market_stats = (

    obs_df

    .groupby("Timestamp")

    .agg(

        AvgMarketMove=("PctChange","mean"),

        AvgAbsMove=("PctChange",
                    lambda x:x.abs().mean()),

        MarketVolatility=("PctChange","std")

    )

    .reset_index()

)

obs_df = obs_df.merge(

    market_stats,

    on="Timestamp",

    how="left"

)

# ----------------------------------------------------------
# Peer Comparison
# ----------------------------------------------------------

peer_stats = (

    obs_df

    .groupby("Timestamp")

    .agg(

        PeerAverage=("ReutersPrice","mean"),

        PeerStd=("ReutersPrice","std")

    )

    .reset_index()

)

obs_df = obs_df.merge(

    peer_stats,

    on="Timestamp",

    how="left"

)

obs_df["PeerDeviation"] = (

    abs(

        obs_df["ReutersPrice"]

        -

        obs_df["PeerAverage"]

    )

    /

    obs_df["PeerAverage"]

)

# ----------------------------------------------------------
# Vendor Consensus
# ----------------------------------------------------------

obs_df["VendorAgreement"] = np.where(

    obs_df["SourceDifferencePct"]<0.002,

    "Strong",

    np.where(

        obs_df["SourceDifferencePct"]<0.005,

        "Moderate",

        "Weak"

    )

)

# ----------------------------------------------------------
# Market Event Indicator
# ----------------------------------------------------------

obs_df["MarketWideMove"] = np.where(

    obs_df["AvgAbsMove"]>0.01,

    True,

    False

)

# ----------------------------------------------------------
# Historical Similarity
# ----------------------------------------------------------

history_lookup = (

    historical_df

    .groupby(

        ["RiskFactor","AlertType"]

    )

    .agg(

        HistoricalCount=("AlertID","count"),

        MostCommonRootCause=(

            "RootCause",

            lambda x:x.mode().iloc[0]

        )

    )

    .reset_index()

)

# Simplify alert type

obs_df["PrimaryAlert"] = (

    obs_df["AlertTypes"]

    .str.split("|")

    .str[0]

    .str.strip()

)

mapping = {

    "Stale Price":"Stale",

    "Large Price Movement":"Outlier",

    "Statistical Outlier":"Outlier",

    "Source Divergence":"Source Divergence",

    "High Feed Latency":"Missing"

}

obs_df["PrimaryAlert"] = (

    obs_df["PrimaryAlert"]

    .replace(mapping)

)

obs_df = obs_df.merge(

    history_lookup,

    left_on=["RiskFactor","PrimaryAlert"],

    right_on=["RiskFactor","AlertType"],

    how="left"

)

# ----------------------------------------------------------
# Confidence Score
# ----------------------------------------------------------

confidence = (

      obs_df["HasRuleAlert"].astype(int)*30

    + obs_df["AIAnomaly"].astype(int)*25

    + (obs_df["VendorAgreement"]=="Weak").astype(int)*20

    + (obs_df["PeerDeviation"]>0.01).astype(int)*15

    + obs_df["HistoricalCount"].fillna(0).clip(0,5)*2

)

obs_df["ConfidenceScore"] = confidence.clip(0,100)

# ----------------------------------------------------------
# Recommendation
# ----------------------------------------------------------

def recommendation(row):

    if row["ConfidenceScore"]>=80:
        return "Escalate Immediately"

    if row["ConfidenceScore"]>=60:
        return "Investigate"

    if row["MarketWideMove"]:
        return "Likely Genuine Market Move"

    if row["VendorAgreement"]=="Weak":
        return "Verify Vendor Feed"

    return "Monitor"

obs_df["Recommendation"] = obs_df.apply(

    recommendation,

    axis=1

)

# ----------------------------------------------------------
# Merge back to portfolio level
# ----------------------------------------------------------

merge_cols = [

    "Timestamp",

    "RiskFactor",

    "AvgMarketMove",

    "AvgAbsMove",

    "MarketVolatility",

    "PeerAverage",

    "PeerDeviation",

    "VendorAgreement",

    "MarketWideMove",

    "HistoricalCount",

    "MostCommonRootCause",

    "ConfidenceScore",

    "Recommendation"

]

market_df = market_df.merge(

    obs_df[merge_cols],

    on=["Timestamp","RiskFactor"],

    how="left"

)

# ----------------------------------------------------------
# Dashboard Tables
# ----------------------------------------------------------

summary = pd.DataFrame({

    "Metric":[

        "Total Observations",

        "Average Market Move",

        "Average Confidence",

        "Weak Vendor Agreement",

        "Market Wide Events"

    ],

    "Value":[

        len(obs_df),

        round(obs_df["AvgAbsMove"].mean(),5),

        round(obs_df["ConfidenceScore"].mean(),1),

        (obs_df["VendorAgreement"]=="Weak").sum(),

        obs_df["MarketWideMove"].sum()

    ]

})

display(summary)

# ----------------------------------------------------------
# Top Alerts
# ----------------------------------------------------------

top = (

    obs_df

    .sort_values(

        "ConfidenceScore",

        ascending=False

    )

    .head(20)

)

display(

top[

[

"Timestamp",

"RiskFactor",

"AlertTypes",

"ConfidenceScore",

"VendorAgreement",

"PeerDeviation",

"HistoricalCount",

"MostCommonRootCause",

"Recommendation"

]

]

)

# ----------------------------------------------------------
# Visualisation
# ----------------------------------------------------------

fig = px.scatter(

    obs_df,

    x="PeerDeviation",

    y="ConfidenceScore",

    color="Recommendation",

    hover_data=[

        "RiskFactor",

        "AlertTypes"

    ],

    title="Alert Confidence vs Peer Deviation"

)

fig.show()

# ----------------------------------------------------------
# Confidence Distribution
# ----------------------------------------------------------

fig = px.histogram(

    obs_df,

    x="ConfidenceScore",

    nbins=20,

    title="Confidence Score Distribution"

)

fig.show()

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

market_df.to_csv(

    PROJECT_ROOT /

    "data/processed/market_data_with_context.csv",

    index=False

)

obs_df.to_csv(

    PROJECT_ROOT /

    "data/processed/context_observations.csv",

    index=False

)

print()

print("="*70)

print("BLOCK 6 COMPLETED")

print("="*70)

print()

print("Average Confidence:",

round(obs_df["ConfidenceScore"].mean(),1))

print()

print("Next Block: VaR Impact & Business Prioritisation")

BLOCK 6 - MARKET CONTEXT
Loaded records: 14400
Unique market observations: 3600


,Metric,Value
0,Total Observations,3600.00000
1,Average Market Move,0.00142
2,Average Confidence,34.50000
3,Weak Vendor Agreement,15.00000
4,Market Wide Events,150.00000


,Timestamp,RiskFactor,AlertTypes,ConfidenceScore,VendorAgreement,PeerDeviation,HistoricalCount,MostCommonRootCause,Recommendation
2373,2026-06-27 02:07:28.191944,USDJPY,Large Price Movement | Statistical Outlier | S...,100.0,Weak,3.873947,5.0,Bad Mapping,Escalate Immediately
2282,2026-06-23 07:07:28.191944,USDJPY,Large Price Movement | Statistical Outlier | S...,100.0,Weak,3.873569,5.0,Bad Mapping,Escalate Immediately
2272,2026-06-22 21:07:28.191944,USDJPY,Large Price Movement | Statistical Outlier | S...,100.0,Weak,3.870722,5.0,Bad Mapping,Escalate Immediately
2933,2026-06-20 10:07:28.191944,USDSGD,Large Price Movement | Statistical Outlier | S...,94.0,Weak,0.955446,2.0,Missing Feed,Escalate Immediately
3031,2026-06-24 12:07:28.191944,USDSGD,Large Price Movement | Statistical Outlier | S...,94.0,Weak,0.954620,2.0,Missing Feed,Escalate Immediately
3218,2026-07-02 07:07:28.191944,USDSGD,Large Price Movement | Statistical Outlier | S...,94.0,Weak,0.953418,2.0,Missing Feed,Escalate Immediately
911,2026-06-26 04:07:28.191944,EURUSD,Large Price Movement | Statistical Outlier | S...,92.0,Weak,0.962075,1.0,Bad Mapping,Escalate Immediately
1309,2026-07-12 18:07:28.191944,EURUSD,Large Price Movement | Statistical Outlier | S...,92.0,Weak,0.962600,1.0,Bad Mapping,Escalate Immediately
1554,2026-06-22 23:07:28.191944,GBPUSD,Large Price Movement | Statistical Outlier | S...,90.0,Weak,0.958100,NaN,NaN,Escalate Immediately
1645,2026-06-26 18:07:28.191944,GBPUSD,Large Price Movement | Statistical Outlier | S...,90.0,Weak,0.957660,NaN,NaN,Escalate Immediately



BLOCK 6 COMPLETED

Average Confidence: 34.5

Next Block: VaR Impact & Business Prioritisation


In [ ]:
# ============================================================
# BLOCK 7
#
# VaR IMPACT & BUSINESS PRIORITISATION
#
# PURPOSE
# -------
# Convert technical alerts into business risk by estimating:
#
# • Shocked Market Price
# • Estimated P&L Impact
# • VaR Impact
# • Business Impact Score
# • Final Alert Priority
#
# Output:
# market_data_prioritised.csv
# ============================================================

print("="*75)
print("BLOCK 7 - BUSINESS IMPACT")
print("="*75)

# ----------------------------------------------------------
# Load Context Output
# ----------------------------------------------------------

market_df = pd.read_csv(

    PROJECT_ROOT /
    "data/processed/market_data_with_context.csv",

    parse_dates=["Timestamp"]

)

scenario_df = pd.read_csv(

    PROJECT_ROOT /
    "data/raw/scenarios.csv"

)

print("Loaded records:",len(market_df))

# ----------------------------------------------------------
# Assign Scenario
# ----------------------------------------------------------
#
# For demo purposes we randomly assign a scenario.
# Production systems would use the official VaR engine.

market_df["ScenarioID"] = np.random.choice(

    scenario_df["ScenarioID"],

    len(market_df)

)

market_df = market_df.merge(

    scenario_df,

    on="ScenarioID",

    how="left"

)

# ----------------------------------------------------------
# Shocked Market Price
# ----------------------------------------------------------

market_df["ShockedPrice"] = (

    market_df["ReutersPrice"]

    *

    (

        1 +

        market_df["ShockPercent"]/100

    )

)

# ----------------------------------------------------------
# Estimated P&L
# ----------------------------------------------------------
#
# Simple approximation:
#
# PnL = Exposure × Price Change %

market_df["EstimatedPnL"] = (

    market_df["ExposureUSD"]

    *

    (

        market_df["ShockedPrice"]

        -

        market_df["ReutersPrice"]

    )

    /

    market_df["ReutersPrice"]

)

market_df["EstimatedPnL"] = (

    market_df["EstimatedPnL"]

    .round(2)

)

# ----------------------------------------------------------
# Estimated VaR Impact
# ----------------------------------------------------------

market_df["EstimatedVaRImpact"] = (

    abs(

        market_df["EstimatedPnL"]

    )

    *

    market_df["VaRContribution"]

    /100

)

# ----------------------------------------------------------
# Business Impact Score
# ----------------------------------------------------------

impact = (

      market_df["RuleSeverityScore"]*0.25

    + market_df["AISeverityScore"]*0.20

    + market_df["ConfidenceScore"]*0.20

    + (

        market_df["VaRContribution"]

        *4

      )

    + (

        market_df["ExposureUSD"]

        /

        market_df["ExposureUSD"].max()

      )*20

)

market_df["BusinessImpactScore"] = (

    impact

    .clip(0,100)

    .round(1)

)

# ----------------------------------------------------------
# Final Priority
# ----------------------------------------------------------

conditions = [

    market_df["BusinessImpactScore"]>=85,

    market_df["BusinessImpactScore"]>=70,

    market_df["BusinessImpactScore"]>=50,

    market_df["BusinessImpactScore"]>=30

]

labels = [

    "Critical",

    "High",

    "Medium",

    "Low"

]

market_df["FinalPriority"] = np.select(

    conditions,

    labels,

    default="Monitor"

)

# ----------------------------------------------------------
# Business Recommendation
# ----------------------------------------------------------

def recommendation(row):

    if row["FinalPriority"]=="Critical":

        return "Escalate to Market Risk immediately"

    if row["FinalPriority"]=="High":

        return "Investigate within 1 hour"

    if row["FinalPriority"]=="Medium":

        return "Review today"

    if row["FinalPriority"]=="Low":

        return "Monitor"

    return "No action"

market_df["BusinessRecommendation"] = (

    market_df.apply(

        recommendation,

        axis=1

    )

)

# ----------------------------------------------------------
# Executive Dashboard
# ----------------------------------------------------------

summary = pd.DataFrame({

    "Metric":[

        "Total Alerts",

        "Critical",

        "High",

        "Medium",

        "Low",

        "Average Estimated P&L",

        "Average Business Score"

    ],

    "Value":[

        len(market_df),

        (market_df["FinalPriority"]=="Critical").sum(),

        (market_df["FinalPriority"]=="High").sum(),

        (market_df["FinalPriority"]=="Medium").sum(),

        (market_df["FinalPriority"]=="Low").sum(),

        round(

            market_df["EstimatedPnL"].mean(),

            2

        ),

        round(

            market_df["BusinessImpactScore"].mean(),

            1

        )

    ]

})

display(summary)

# ----------------------------------------------------------
# Top Business Risks
# ----------------------------------------------------------

top = (

    market_df

    .sort_values(

        "BusinessImpactScore",

        ascending=False

    )

    .head(20)

)

display(

top[

[

"Timestamp",

"Portfolio",

"RiskFactor",

"Scenario",

"ExposureUSD",

"EstimatedPnL",

"EstimatedVaRImpact",

"BusinessImpactScore",

"FinalPriority",

"BusinessRecommendation"

]

]

)

# ----------------------------------------------------------
# Priority Distribution
# ----------------------------------------------------------

fig = px.bar(

    market_df

    .groupby(

        "FinalPriority"

    )

    .size()

    .reset_index(name="Count"),

    x="FinalPriority",

    y="Count",

    color="FinalPriority",

    text="Count",

    title="Final Business Priority"

)

fig.show()

# ----------------------------------------------------------
# P&L vs Score
# ----------------------------------------------------------

fig = px.scatter(

    market_df.sample(1500),

    x="EstimatedPnL",

    y="BusinessImpactScore",

    color="FinalPriority",

    hover_data=[

        "Portfolio",

        "RiskFactor"

    ],

    title="Business Impact vs Estimated P&L"

)

fig.show()

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

market_df.to_csv(

    PROJECT_ROOT /

    "data/processed/market_data_prioritised.csv",

    index=False

)

print()

print("="*75)

print("BLOCK 7 COMPLETED")

print("="*75)

print()

print("Next Block:")

print("LLM Investigation Summary")

BLOCK 7 - BUSINESS IMPACT
Loaded records: 14400


,Metric,Value
0,Total Alerts,14400.00
1,Critical,2031.00
2,High,2601.00
3,Medium,3706.00
4,Low,3973.00
5,Average Estimated P&L,31868.04
6,Average Business Score,56.40


,Timestamp,Portfolio,RiskFactor,Scenario,ExposureUSD,EstimatedPnL,EstimatedVaRImpact,BusinessImpactScore,FinalPriority,BusinessRecommendation
5788,2026-06-18 12:07:28.191944,Corporate Banking,GBPUSD,FX Crisis,49664543,1825166.44,280345.565184,100.0,Critical,Escalate to Market Risk immediately
12129,2026-06-24 13:07:28.191944,FX Trading,USDSGD,China Slowdown,28340704,-1991643.34,281817.532610,100.0,Critical,Escalate to Market Risk immediately
3650,2026-06-26 05:07:28.191944,Private Banking,EURUSD,Pandemic,31247282,-1640684.50,219031.380750,100.0,Critical,Escalate to Market Risk immediately
3646,2026-06-26 04:07:28.191944,Private Banking,EURUSD,Interest Rate Shock,31247282,1646290.19,219779.740365,100.0,Critical,Escalate to Market Risk immediately
3647,2026-06-26 04:07:28.191944,Treasury,EURUSD,Interest Rate Shock,12897687,679525.84,76242.799248,100.0,Critical,Escalate to Market Risk immediately
5238,2026-07-12 18:07:28.191944,Private Banking,EURUSD,China Slowdown,31247282,-2195903.15,293153.070525,100.0,Critical,Escalate to Market Risk immediately
5239,2026-07-12 18:07:28.191944,Treasury,EURUSD,Inflation Spike,12897687,-403556.35,45279.022470,100.0,Critical,Escalate to Market Risk immediately
5242,2026-07-12 19:07:28.191944,Private Banking,EURUSD,Oil Spike,31247282,-1452731.71,193939.683285,100.0,Critical,Escalate to Market Risk immediately
5768,2026-06-18 07:07:28.191944,Corporate Banking,GBPUSD,Geopolitical Event,49664543,-617017.40,94773.872640,100.0,Critical,Escalate to Market Risk immediately
5771,2026-06-18 07:07:28.191944,Treasury,GBPUSD,Fed Rate Cut,7046416,449031.12,87920.293296,100.0,Critical,Escalate to Market Risk immediately



BLOCK 7 COMPLETED

Next Block:
LLM Investigation Summary


In [ ]:
# ============================================================
# BLOCK 8
#
# ALERT INVESTIGATION PACK
#
# PURPOSE
# -------
# Consolidate all evidence into a structured investigation
# package ready for the LLM.
#
# The LLM will NEVER analyse raw data.
# Instead it will receive only this evidence package.
#
# Output
# ------
# investigation_pack.csv
# ============================================================

print("="*75)
print("BLOCK 8 - ALERT INVESTIGATION PACK")
print("="*75)

# ----------------------------------------------------------
# Load prioritised alerts
# ----------------------------------------------------------

market_df = pd.read_csv(

    PROJECT_ROOT /
    "data/processed/market_data_prioritised.csv",

    parse_dates=["Timestamp"]

)

print("Loaded records:", len(market_df))

# ----------------------------------------------------------
# Keep only alerts
# ----------------------------------------------------------

alerts = market_df[

    market_df["CombinedAlert"]

].copy()

print("Alerts:", len(alerts))

# ----------------------------------------------------------
# Determine Primary Root Cause
# ----------------------------------------------------------

def determine_root_cause(row):

    if row["Rule_Stale"]:
        return "Possible stale market data"

    if row["Rule_SourceDivergence"]:
        return "Vendor price disagreement"

    if row["Rule_HighLatency"]:
        return "Delayed market data feed"

    if row["Rule_ZScoreOutlier"]:
        return "Extreme statistical movement"

    if row["Rule_LargeMove"]:
        return "Large market movement"

    if row["AIAnomaly"]:
        return "Multivariate statistical anomaly"

    return "Unknown"

alerts["LikelyRootCause"] = alerts.apply(

    determine_root_cause,

    axis=1

)

# ----------------------------------------------------------
# Suggested Next Action
# ----------------------------------------------------------

def next_action(row):

    if row["FinalPriority"]=="Critical":

        return (
            "Immediately contact Market Data "
            "Operations and Market Risk."
        )

    if row["Rule_SourceDivergence"]:

        return (
            "Compare Reuters and Bloomberg."
        )

    if row["Rule_HighLatency"]:

        return (
            "Check feed latency and infrastructure."
        )

    if row["Rule_Stale"]:

        return (
            "Verify vendor refresh schedule."
        )

    if row["MarketWideMove"]:

        return (
            "Check external market news."
        )

    return "Review manually."

alerts["NextAction"] = alerts.apply(

    next_action,

    axis=1

)

# ----------------------------------------------------------
# Build Investigation JSON
# ----------------------------------------------------------

def build_pack(row):

    return {

        "alert_id":row["AlertID"],

        "timestamp":str(row["Timestamp"]),

        "portfolio":row["Portfolio"],

        "risk_factor":row["RiskFactor"],

        "scenario":row["Scenario"],

        "priority":row["FinalPriority"],

        "business_score":

            float(row["BusinessImpactScore"]),

        "confidence":

            float(row["ConfidenceScore"]),

        "estimated_pnl":

            float(row["EstimatedPnL"]),

        "estimated_var":

            float(row["EstimatedVaRImpact"]),

        "exposure":

            float(row["ExposureUSD"]),

        "rule_alert":

            bool(row["HasRuleAlert"]),

        "ai_alert":

            bool(row["AIAnomaly"]),

        "rule_types":

            row["AlertTypes"],

        "rule_evidence":

            row["RuleEvidence"],

        "vendor_agreement":

            row["VendorAgreement"],

        "peer_deviation":

            float(row["PeerDeviation"]),

        "market_wide":

            bool(row["MarketWideMove"]),

        "historical_root_cause":

            row["MostCommonRootCause"],

        "likely_root_cause":

            row["LikelyRootCause"],

        "recommendation":

            row["BusinessRecommendation"],

        "next_action":

            row["NextAction"]

    }

alerts["InvestigationPack"] = alerts.apply(

    build_pack,

    axis=1

)

# ----------------------------------------------------------
# Save JSON string
# ----------------------------------------------------------

alerts["InvestigationJSON"] = alerts[
    "InvestigationPack"
].apply(

    lambda x: json.dumps(
        x,
        indent=2
    )

)

# ----------------------------------------------------------
# Executive Summary
# ----------------------------------------------------------

summary = pd.DataFrame({

    "Metric":[

        "Investigation Packs",

        "Critical",

        "High",

        "Average Confidence",

        "Average Business Score"

    ],

    "Value":[

        len(alerts),

        (alerts["FinalPriority"]=="Critical").sum(),

        (alerts["FinalPriority"]=="High").sum(),

        round(alerts["ConfidenceScore"].mean(),1),

        round(alerts["BusinessImpactScore"].mean(),1)

    ]

})

display(summary)

# ----------------------------------------------------------
# Sample Investigation Pack
# ----------------------------------------------------------

sample = alerts.iloc[0]

print("="*75)
print("SAMPLE INVESTIGATION PACK")
print("="*75)

print(sample["InvestigationJSON"])

# ----------------------------------------------------------
# Dashboard
# ----------------------------------------------------------

display(

alerts[

[

"AlertID",

"Portfolio",

"RiskFactor",

"FinalPriority",

"BusinessImpactScore",

"ConfidenceScore",

"LikelyRootCause",

"NextAction"

]

].head(20)

)

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

alerts.to_csv(

    PROJECT_ROOT /

    "data/processed/investigation_pack.csv",

    index=False

)

print()

print("="*75)

print("BLOCK 8 COMPLETED")

print("="*75)

print()

print("Investigation Packs created:", len(alerts))

print()

print("Next Block: LLM Investigation Report")

BLOCK 8 - ALERT INVESTIGATION PACK
Loaded records: 14400
Alerts: 7556


,Metric,Value
0,Investigation Packs,7556.0
1,Critical,1362.0
2,High,1581.0
3,Average Confidence,52.2
4,Average Business Score,61.7


SAMPLE INVESTIGATION PACK
{
  "alert_id": "RULE_AUDUSD_20260618_050728",
  "timestamp": "2026-06-18 05:07:28.191944",
  "portfolio": "Corporate Banking",
  "risk_factor": "AUDUSD",
  "scenario": "Oil Spike",
  "priority": "High",
  "business_score": 72.7,
  "confidence": 53.0,
  "estimated_pnl": -1878862.0,
  "estimated_var": 197656.2824,
  "exposure": 40413058.0,
  "rule_alert": true,
  "ai_alert": false,
  "rule_types": "High Feed Latency",
  "rule_evidence": "Feed latency 322 seconds",
  "vendor_agreement": "Strong",
  "peer_deviation": 0.9789811540187352,
  "market_wide": false,
  "historical_root_cause": "Market Movement",
  "likely_root_cause": "Delayed market data feed",
  "recommendation": "Investigate within 1 hour",
  "next_action": "Check feed latency and infrastructure."
}


,AlertID,Portfolio,RiskFactor,FinalPriority,BusinessImpactScore,ConfidenceScore,LikelyRootCause,NextAction
0,RULE_AUDUSD_20260618_050728,Corporate Banking,AUDUSD,High,72.7,53.0,Delayed market data feed,Check feed latency and infrastructure.
1,RULE_AUDUSD_20260618_050728,FX Trading,AUDUSD,Low,34.1,53.0,Delayed market data feed,Check feed latency and infrastructure.
2,RULE_AUDUSD_20260618_050728,Private Banking,AUDUSD,Low,35.3,53.0,Delayed market data feed,Check feed latency and infrastructure.
3,RULE_AUDUSD_20260618_050728,Treasury,AUDUSD,Monitor,27.6,53.0,Delayed market data feed,Check feed latency and infrastructure.
4,RULE_AUDUSD_20260618_060728,Corporate Banking,AUDUSD,High,75.2,53.0,Delayed market data feed,Check feed latency and infrastructure.
5,RULE_AUDUSD_20260618_060728,FX Trading,AUDUSD,Low,36.6,53.0,Delayed market data feed,Check feed latency and infrastructure.
6,RULE_AUDUSD_20260618_060728,Private Banking,AUDUSD,Low,37.8,53.0,Delayed market data feed,Check feed latency and infrastructure.
7,RULE_AUDUSD_20260618_060728,Treasury,AUDUSD,Low,30.1,53.0,Delayed market data feed,Check feed latency and infrastructure.
12,RULE_AUDUSD_20260618_080728,Corporate Banking,AUDUSD,High,74.1,53.0,Delayed market data feed,Check feed latency and infrastructure.
13,RULE_AUDUSD_20260618_080728,FX Trading,AUDUSD,Low,35.6,53.0,Delayed market data feed,Check feed latency and infrastructure.



BLOCK 8 COMPLETED

Investigation Packs created: 7556

Next Block: LLM Investigation Report


In [ ]:
from openai import OpenAI
import os
import time
import pandas as pd

print("="*75)
print("BLOCK 9 - GPT INVESTIGATION REPORT")
print("="*75)

# ---------------------------------------------------------
# Create OpenAI client
# ---------------------------------------------------------
from google.colab import userdata

# Retrieve the API key from Colab's user data secrets
openai_api_key = userdata.get('OPENAI_API_KEY')

client = OpenAI(
    api_key=openai_api_key
)

# ---------------------------------------------------------
# Load Investigation Packs
# ---------------------------------------------------------

alerts = pd.read_csv(

    PROJECT_ROOT /
    "data/processed/investigation_pack.csv"

)

print("Alerts loaded:",len(alerts))

# ---------------------------------------------------------
# LLM Function
# ---------------------------------------------------------

SYSTEM_PROMPT = """

You are a Senior Market Risk Officer at a global investment bank.

Your responsibility is to investigate market data quality alerts that may affect

valuation, P&L, Value at Risk (VaR), regulatory reporting and trading decisions.

You are provided ONLY with a structured Investigation Pack generated by the bank's

AI Control Framework.

Use ONLY the supplied evidence.

Do NOT invent facts, assumptions or market events.

If evidence is insufficient, explicitly state that additional investigation is required.

Your audience is:

• Head of Market Risk

• Market Risk Operations

• Trading Desk Supervisors

• Technology Support

Write in a concise executive style.

Your report must explain WHY the alert matters to the business, not merely repeat

the input values.

Where appropriate, translate technical findings into business language.

For example:

- Explain why stale prices matter.

- Explain why feed latency impacts VaR.

- Explain why vendor disagreement increases operational risk.

- Explain why a market-wide move may indicate a genuine market event instead of bad data.

Avoid simply repeating numerical values unless they materially support the conclusion.

If numerical values are important:

• round large numbers appropriately

• describe materiality

• avoid listing every field from the investigation pack

Return the report using EXACTLY the following sections.

## Executive Assessment

Provide a 3–5 sentence executive summary explaining:

- what happened

- why it matters

- urgency

- overall business impact

## Likely Root Cause

Explain the most likely cause using only the supplied evidence.

Mention contradictory evidence if relevant.

State whether confidence is High, Moderate or Low.

## Business Impact Assessment

Describe the potential impact on:

- valuation

- P&L

- VaR

- trading decisions

- regulatory reporting

- operational risk

Do not exaggerate.

## Recommended Actions

Provide 3–5 prioritized actions.

The first recommendation should always be the highest priority.

Separate Immediate actions from Follow-up actions where appropriate.

## Overall Assessment

Provide:

Priority:

(Critical / High / Medium / Low)

AI Confidence:

(Very High / High / Moderate / Low / Very Low)

Final Recommendation:

One concise sentence suitable for an executive dashboard.

Maximum length: 250 words.

Never mention prompts, AI models, JSON, Investigation Pack or internal implementation.

"""


def generate_report(investigation_json):

    prompt = f"""
Investigation Pack

{investigation_json}
"""

    try:

        response = client.responses.create(

            model="gpt-5-mini",

            input=[

                {
                    "role":"system",
                    "content":SYSTEM_PROMPT
                },

                {
                    "role":"user",
                    "content":prompt
                }

            ]

        )

        return response.output_text

    except Exception as e:

        return f"LLM Error: {e}"


# ---------------------------------------------------------
# Generate Reports
# ---------------------------------------------------------

reports=[]

TOTAL=min(20,len(alerts))

print(f"Generating {TOTAL} reports...\n")

for i,row in alerts.head(TOTAL).iterrows():

    print(f"{i+1}/{TOTAL}")

    report = generate_report(

        row["InvestigationJSON"]

    )

    reports.append(report)

    time.sleep(0.5)

alerts = alerts.head(TOTAL).copy()

alerts["GPTReport"]=reports

# ---------------------------------------------------------
# Display Sample
# ---------------------------------------------------------

print("="*75)
print("SAMPLE GPT REPORT")
print("="*75)

print(alerts.iloc[0]["GPTReport"])

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

alerts.to_csv(

    PROJECT_ROOT /

    "reports/gpt_investigation_reports.csv",

    index=False

)

print()

print("="*75)
print("BLOCK 9 COMPLETED")
print("="*75)

print()

print("Reports Generated:",len(alerts))

BLOCK 9 - GPT INVESTIGATION REPORT
Alerts loaded: 7556
Generating 20 reports...

1/20
2/20
3/20
4/20
5/20
6/20
7/20
8/20
9/20
10/20
11/20
12/20
13/20
14/20
15/20
16/20
17/20
18/20
19/20
20/20
SAMPLE GPT REPORT
## Executive Assessment

A high-priority alert flagged unusually high feed latency for AUDUSD (322 seconds), coinciding with an estimated negative P&L impact of ~‑1.9m and portfolio exposure ~40.4m. Vendor agreement is strong and the issue is not market-wide, so this appears to be delayed/latent market data rather than a genuine market move. Urgent investigation is required because 5+ minute stale prices can produce incorrect valuations, misstate P&L and VaR, and cause unsafe trading decisions. Immediate action should be taken within the hour to contain risk.

## Likely Root Cause

Most likely cause: delayed market data feed / infrastructure latency (rule: High Feed Latency; evidence: 322s feed delay). Supporting evidence: strong vendor agreement and non-market-wide flag — prices

In [ ]:
# STREAMLIT

In [ ]:
!pip -q install streamlit pyngrok plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 103.1 MB/s eta 0:00:00


In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok

ngrok.set_auth_token("2ykyRPU0jxUoqpoiCFqt6f8fiuE_49jt3Nb9uUnD3v4GBMU2F")

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px
import os

st.set_page_config(
    page_title="AI Market Risk Control Cockpit",
    page_icon="🏦",
    layout="wide"
)

DATA_PATH="/content/drive/MyDrive/Hackathon/var_control_cockpit"

priority = pd.read_csv(

    f"{DATA_PATH}/data/processed/market_data_prioritised.csv"

)

reports = pd.read_csv(

    f"{DATA_PATH}/reports/gpt_investigation_reports.csv"

)

# ---------------------------------------
# HEADER
# ---------------------------------------

st.title("🏦 AI Market Risk Control Cockpit")
st.caption("AI Assisted Market Data Quality Monitoring")

st.divider()

# ---------------------------------------
# KPIs
# ---------------------------------------

critical=(priority["FinalPriority"]=="Critical").sum()
high=(priority["FinalPriority"]=="High").sum()

total=len(priority)

total_pnl=priority["EstimatedPnL"].sum()

c1,c2,c3,c4=st.columns(4)

c1.metric("Critical Alerts",critical)
c2.metric("High Alerts",high)
c3.metric("Total Alerts",total)
c4.metric(
    "Estimated P&L",
    f"${total_pnl:,.0f}"
)

st.divider()

# ---------------------------------------
# ALERT DISTRIBUTION
# ---------------------------------------

left,right=st.columns(2)

with left:

    fig=px.histogram(
        priority,
        x="FinalPriority",
        color="FinalPriority",
        title="Alert Priority Distribution"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

with right:

    fig2=px.scatter(
        priority,
        x="BusinessImpactScore",
        y="EstimatedPnL",
        color="FinalPriority",
        hover_data=["RiskFactor","Portfolio"],
        title="Business Impact vs Estimated P&L"
    )

    st.plotly_chart(
        fig2,
        use_container_width=True
    )

st.divider()

# ---------------------------------------
# TOP ALERTS
# ---------------------------------------

st.subheader("🚨 Highest Priority Alerts")

display_cols=[
    "AlertID",
    "Portfolio",
    "RiskFactor",
    "FinalPriority",
    "BusinessImpactScore",
    "EstimatedPnL",
    "ConfidenceScore"
]

top=priority.sort_values(
    "BusinessImpactScore",
    ascending=False
)

st.dataframe(
    top[display_cols],
    use_container_width=True,
    height=350
)

st.divider()

# ---------------------------------------
# REPORT VIEWER
# ---------------------------------------

st.subheader("🤖 AI Investigation Report")

selected=st.selectbox(
    "Select Alert",
    reports["AlertID"]
)

report=reports.loc[
    reports.AlertID==selected,
    "GPTReport"
].iloc[0]

st.markdown(report)

st.divider()

# ---------------------------------------
# FEED LATENCY
# ---------------------------------------

st.subheader("Feed Latency")

fig3=px.box(
    priority,
    x="Portfolio",
    y="FeedLatencySeconds",
    color="FinalPriority",
    title="Feed Latency by Portfolio"
)

st.plotly_chart(
    fig3,
    use_container_width=True
)

st.divider()

# ---------------------------------------
# SEARCH
# ---------------------------------------

st.subheader("Search Alerts")

text=st.text_input("Risk Factor")

if text:

    df=priority[
        priority.RiskFactor.str.contains(
            text,
            case=False
        )
    ]

    st.dataframe(df,use_container_width=True)

Overwriting app.py


In [ ]:
!streamlit run app.py >/content/log.txt 2>&1 &

In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://1b3f-35-236-247-131.ngrok-free.app" -> "http://localhost:8501"
